# DataFrames Basics

## Why DataFrames over RDDs?

DataFrames were introduced in Spark 1.3 and are now the recommended API for structured data. They offer three major advantages over raw RDDs:

| Feature | RDD | DataFrame |
|---------|-----|-----------|
| **Schema** | None — just typed collections | Named, typed columns with an explicit schema |
| **Optimizer** | None — runs exactly what you write | **Catalyst optimizer** rewrites your query for efficiency |
| **SQL** | Not available | Full SQL support via `spark.sql()` |
| **Performance** | JVM overhead for Python lambdas | Vectorized execution in the JVM, often 10-100x faster |
| **Readability** | Functional chains | Declarative, table-like operations |

Under the hood, a DataFrame *is* an RDD of `Row` objects, but Spark can optimize it because it knows the schema ahead of time.

In this notebook we will:
- Create a DataFrame from a Python list with an explicit schema
- Inspect structure with `show()`, `printSchema()`, and `dtypes`
- Apply selections, filters, and column transformations
- Use `pyspark.sql.functions` for expressive column expressions
- Generate summary statistics with `describe()`

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("03-DataFrames-Basics")
    .getOrCreate()
)




SparkSession ready. Version: 3.5.3


In [14]:
print("SparkSession ready. Version:", spark.version)
print("Master URL             :", spark.sparkContext.master)
print("Default parallelism    :", spark.sparkContext.defaultParallelism)

SparkSession ready. Version: 3.5.3
Master URL             : spark://spark-master:7077
Default parallelism    : 4


In [18]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define an explicit schema using StructType.
# This is best practice — it avoids the cost of schema inference
# and makes the expected shape of the data self-documenting.
schema = StructType([
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=True),
    StructField("age",        IntegerType(), nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
])

# Raw data as a list of tuples — order must match the schema definition above
data = [
    ("Alice",   "Engineering", 30, 95000.0),
    ("Bob",     "Marketing",   45, 72000.0),
    ("Carol",   "Engineering", 27, 88000.0),
    ("David",   "HR",          33, 65000.0),
    ("Eve",     "Engineering", 25, 91000.0),
    ("Frank",   "Marketing",   38, 78000.0),
    ("Grace",   "HR",          29, 67000.0),
    ("Hector",  "Engineering", 42, 105000.0),
]

# createDataFrame() distributes the data and binds the schema
df = spark.createDataFrame(data, schema=schema)

print("DataFrame created with", df.count(), "rows.")

DataFrame created with 8 rows.


In [21]:
# .show() — print a tabular preview of the data (default 20 rows)
# truncate=False ensures long values are not cut off
print("── df.show() ──")
df.show(truncate=False)

# .printSchema() — display the tree-structured schema with types and nullability
print("── df.printSchema() ──")
df.printSchema()

# .dtypes — Python list of (column_name, type_string) tuples
print("── df.dtypes ──")
for col_name, col_type in df.dtypes:
    print(f"  {col_name:<12} -> {col_type}")

# .count() — total number of rows (triggers a job)
print("\nTotal rows:", df.count())

── df.show() ──
+------+-----------+---+--------+
|name  |department |age|salary  |
+------+-----------+---+--------+
|Alice |Engineering|30 |95000.0 |
|Bob   |Marketing  |45 |72000.0 |
|Carol |Engineering|27 |88000.0 |
|David |HR         |33 |65000.0 |
|Eve   |Engineering|25 |91000.0 |
|Frank |Marketing  |38 |78000.0 |
|Grace |HR         |29 |67000.0 |
|Hector|Engineering|42 |105000.0|
+------+-----------+---+--------+

── df.printSchema() ──
root
 |-- name: string (nullable = false)
 |-- department: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: double (nullable = true)

── df.dtypes ──
  name         -> string
  department   -> string
  age          -> int
  salary       -> double

Total rows: 8


In [19]:
# ── Selecting columns ─────────────────────────────────────────────────────
# .select() returns a new DataFrame with only the specified columns
print("── select(name, age) ──")
df.select("name", "age").show()

# ── Filtering rows ────────────────────────────────────────────────────────
# .filter() (alias: .where()) keeps rows that satisfy a boolean condition
print("── filter(age > 30) ──")
df.filter(df.age > 30).show()

# ── Adding / replacing columns ────────────────────────────────────────────
# .withColumn() returns a new DataFrame with a column added or replaced.
# It does NOT mutate the original DataFrame (DataFrames are immutable).
print("── withColumn: age_next_year ──")
df.withColumn("age_next_year", df.age + 1).select("name", "age", "age_next_year").show()

── select(name, age) ──
+------+---+
|  name|age|
+------+---+
| Alice| 30|
|   Bob| 45|
| Carol| 27|
| David| 33|
|   Eve| 25|
| Frank| 38|
| Grace| 29|
|Hector| 42|
+------+---+

── filter(age > 30) ──
+------+-----------+---+--------+
|  name| department|age|  salary|
+------+-----------+---+--------+
|   Bob|  Marketing| 45| 72000.0|
| David|         HR| 33| 65000.0|
| Frank|  Marketing| 38| 78000.0|
|Hector|Engineering| 42|105000.0|
+------+-----------+---+--------+

── withColumn: age_next_year ──
+------+---+-------------+
|  name|age|age_next_year|
+------+---+-------------+
| Alice| 30|           31|
|   Bob| 45|           46|
| Carol| 27|           28|
| David| 33|           34|
|   Eve| 25|           26|
| Frank| 38|           39|
| Grace| 29|           30|
|Hector| 42|           43|
+------+---+-------------+



## Column Expressions — `col()`, `lit()`, and `F.when()`

Instead of referencing columns directly as `df.column_name`, it is often cleaner to use helper functions from `pyspark.sql.functions`:

| Function | Purpose | Example |
|----------|---------|--------|
| `col("name")` | Reference a column by name string | `col("age") > 30` |
| `lit(value)` | Create a column with a constant literal value | `lit(2024)` |
| `F.when(cond, val).otherwise(val)` | Conditional expression (like SQL CASE WHEN) | `F.when(col("age") >= 35, "senior").otherwise("junior")` |

Using `col()` and `F.*` functions is preferred over `df.column_name` notation because:
- Column references are decoupled from a specific DataFrame object
- They compose cleanly inside complex expressions
- They work inside `.select()` and `.withColumn()` without ambiguity

In [20]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit 

# col() — reference a column by name; works independently of the DataFrame variable
# lit() — wrap a Python scalar into a Spark Column of constant value
# F.when().otherwise() — vectorised conditional, equivalent to SQL CASE WHEN

df_enriched = (
    df
    # Add a constant column (useful for tagging rows with a batch ID, version, etc.)
    .withColumn("cohort_year", lit(2024))

    # Conditional: classify employees as junior or senior based on age
    .withColumn(
        "seniority",
        F.when(col("age") >= 35, "senior").otherwise("junior")
    )

    # Compute salary in thousands (rounded to 1 decimal) using col() expression
    .withColumn(
        "salary_k",
        F.round(col("salary") / lit(1000), 1)
    )
)

df_enriched.select("name", "age", "seniority", "salary_k", "cohort_year").show()

+------+---+---------+--------+-----------+
|  name|age|seniority|salary_k|cohort_year|
+------+---+---------+--------+-----------+
| Alice| 30|   junior|    95.0|       2024|
|   Bob| 45|   senior|    72.0|       2024|
| Carol| 27|   junior|    88.0|       2024|
| David| 33|   junior|    65.0|       2024|
|   Eve| 25|   junior|    91.0|       2024|
| Frank| 38|   senior|    78.0|       2024|
| Grace| 29|   junior|    67.0|       2024|
|Hector| 42|   senior|   105.0|       2024|
+------+---+---------+--------+-----------+



In [22]:
# ── Summary Statistics ────────────────────────────────────────────────────
# .describe() computes count, mean, stddev, min, max for numeric
# and string columns. Returns a new DataFrame — call .show() to display it.

print("── df.describe() ──")
df.describe("age", "salary").show()

# For more precise statistics, use .summary() which also includes percentiles
print("── df.summary() ──")
df.select("age", "salary").summary().show()

── df.describe() ──
+-------+------------------+------------------+
|summary|               age|            salary|
+-------+------------------+------------------+
|  count|                 8|                 8|
|   mean|            33.625|           82625.0|
| stddev|7.2886898685566255|14352.077997876922|
|    min|                25|           65000.0|
|    max|                45|          105000.0|
+-------+------------------+------------------+

── df.summary() ──


+-------+------------------+------------------+
|summary|               age|            salary|
+-------+------------------+------------------+
|  count|                 8|                 8|
|   mean|            33.625|           82625.0|
| stddev|7.2886898685566255|14352.077997876922|
|    min|                25|           65000.0|
|    25%|                27|           67000.0|
|    50%|                30|           78000.0|
|    75%|                38|           91000.0|
|    max|                45|          105000.0|
+-------+------------------+------------------+



In [23]:
# Release cluster resources
spark.stop()
print("SparkSession stopped. All done!")

SparkSession stopped. All done!
